# LetterWorld: Anakin (JAX/TPU) vs SB3 (PyTorch) — return vs wall-clock

This notebook runs a **head-to-head, wall-clock** comparison on the LetterWorld
reward-machine task between two DQN implementations that share *identical*
learning dynamics (architecture, $\gamma$, learning rate, batch size, replay
buffer, target-update interval, $\epsilon$-schedule, and eval protocol):

- **Anakin (JAX)** — a fully-jitted, `vmap`-parallel single-DQN that collects
  rollouts across `NUM_ENVS` environments at once and runs a matched
  `updates_per_rollout` on the TPU.
- **SB3 (PyTorch)** — Stable-Baselines3 DQN on the same task, on the TPU VM's
  **CPU** (PyTorch does not use the TPU here).

**What we demonstrate (Option C):** scaled JAX parallelism *plus* a matched,
raised update ratio. We overlay **both arms on one axis** — eval **return (y)
vs wall-clock seconds (x)** — across **5 seeds**, drawing per-seed curves plus a
mean$\pm$std band. The JAX curve reaching high return far to the **left** (less
wall-clock) is the point.

**Honest framing.** This is *not* a claim that JAX's algorithm is better — the
two arms are algorithmically matched. It is a claim about **throughput**: the
same number of updates and env-steps is cheap for a jitted, parallel JAX kernel
on a TPU and expensive for SB3's Python/PyTorch step loop on CPU. See the
**Fairness** section at the end for the exact terms.

> Set the runtime to **TPU** first: *Runtime → Change runtime type → TPU*.

## 1. Runtime / TPU check

We import JAX and report its default backend and devices. On a TPU Colab
runtime with `jax[tpu]` installed, **JAX selects the TPU as its default backend
automatically** — we do **not** set `JAX_PLATFORMS`, because forcing a platform
here would override that. The trainers below run on whatever the default backend
is.

If no TPU device is present, switch the runtime (*Runtime → Change runtime type
→ TPU*) and re-run from the top.

In [ ]:
import jax

print("jax version:", jax.__version__)
print("default backend:", jax.default_backend())
print("devices:", jax.devices())

_has_tpu = any(d.platform == "tpu" for d in jax.devices())
if _has_tpu:
    print("\nTPU detected — JAX will use it as the default backend.")
else:
    print(
        "\nNo TPU device found. Select a TPU runtime via "
        "Runtime -> Change runtime type -> TPU, then re-run from the top. "
        "(Do NOT set JAX_PLATFORMS here; on a TPU runtime JAX uses the TPU "
        "automatically.)"
    )

## 2. Install dependencies

Order matters: we install **`jax[tpu]` first** so the TPU build of JAX is
resolved, then install the rest **without** letting them downgrade JAX. We
re-print `jax.__version__` and the backend afterwards to confirm nothing was
clobbered.

> **Note:** if Colab preinstalled a conflicting JAX, a one-time **runtime
> restart** may be required after this cell (*Runtime → Restart runtime*), then
> re-run from the top. This is a pip/import-cache issue, not a TPU issue.

In [ ]:
# `_sh` runs a shell command in Colab. Using get_ipython().system(...) instead
# of the `!cmd` shorthand keeps every cell parseable as plain Python.
_sh = get_ipython().system  # noqa: F821  (provided by the Colab/IPython kernel)

# Install the TPU build of JAX first so the correct wheels are resolved.
_sh('pip install -q "jax[tpu]"')

# Then the rest, without downgrading jax (jax/jaxlib are already resolved above).
_sh("pip install -q stoa-env optax stable-baselines3 matplotlib")

# The values below reflect the JAX already imported in THIS kernel. Re-importing
# does NOT pick up a freshly pip-installed jax[tpu]/jaxlib backend — only a
# kernel restart does. If these look stale after the install (e.g. wrong version
# or a non-TPU backend), do Runtime -> Restart runtime, then re-run from the top.
import jax

print("jax version (current kernel):", jax.__version__)
print("default backend (current kernel):", jax.default_backend())

## 3. Clone and install the repo

We shallow-clone the public `experimental/jax` branch of `TristanBester/pycrm`,
put it on `sys.path`, and install it editable so its dependencies resolve. The
LetterWorld trainers live under `examples/rm/letterworld_anakin/`.

In [ ]:
import sys

_sh = get_ipython().system  # noqa: F821  (provided by the Colab/IPython kernel)

_sh(
    "git clone --branch experimental/jax --depth 1 "
    "https://github.com/TristanBester/pycrm.git /content/pycrm"
)

if "/content/pycrm" not in sys.path:
    sys.path.insert(0, "/content/pycrm")

_sh("pip install -q -e /content/pycrm")

## 4. ⚠️ stoa-on-TPU guard (make-or-break)

Before spending time on a full run, we verify that `stoa` imports and that a
trivial JAX op **executes on the TPU device**. Some backends (notably Apple
Metal) reject small-dtype ops with an `UNIMPLEMENTED` error; if that happens on
this TPU runtime the run cannot proceed on the TPU.

**If this guard fails**, the fallback is **CPU** — but JAX binds its backend
**once per kernel, at `import jax`**, so it *cannot be switched mid-session*. To
fall back you must:

1. Add a **new cell at the very top** of the notebook containing
   `import os; os.environ["JAX_PLATFORMS"] = "cpu"` **before any `import jax`**.
2. **Runtime → Restart runtime.**
3. Re-run from the top.

We do **not** attempt to hot-swap to CPU here — that does not work once JAX has
initialized.

In [ ]:
import jax
import jax.numpy as jnp

USE_TPU = False
try:
    import stoa  # noqa: F401

    # Force a trivial op onto the TPU device and materialize the result.
    tpu_dev = next(d for d in jax.devices() if d.platform == "tpu")
    x = jax.device_put(jnp.array(0, dtype=jnp.int8), tpu_dev)
    y = (x + jnp.array(1, dtype=jnp.int8)).block_until_ready()
    assert int(y) == 1
    USE_TPU = True
    print("stoa import OK; trivial int8 op executed on:", y.devices())
    print("USE_TPU =", USE_TPU)
except StopIteration:
    print(
        "No TPU device available — go back to cell 1 and select a TPU runtime "
        "(Runtime -> Change runtime type -> TPU), then re-run from the top."
    )
except Exception as exc:  # noqa: BLE001
    print("stoa-on-TPU guard FAILED:", type(exc).__name__)
    print(exc)
    print(
        "\nFALLBACK (CPU): JAX binds its backend once, at `import jax`, and "
        "cannot be switched mid-session.\n"
        "  1. Add a NEW cell at the VERY TOP of this notebook:\n"
        '         import os; os.environ["JAX_PLATFORMS"] = "cpu"\n'
        "     (it must run BEFORE any `import jax`).\n"
        "  2. Runtime -> Restart runtime.\n"
        "  3. Re-run the notebook from the top.\n"
        "Do NOT expect this cell to hot-swap to CPU in place."
    )

## 5. Configuration

Editable knobs for the comparison. The key coupling is:

```
updates_per_rollout = num_envs * ROLLOUT // env_steps_per_update
```

so scaling `NUM_ENVS` up (to **1024** or **2048**) to *fill the TPU* raises the
number of env-steps collected per rollout — bump `ENV_STEPS_PER_UPDATE` in step
to keep `updates_per_rollout` in a sane range. `ENV_STEPS_PER_UPDATE` is the
**matched** update ratio applied to *both* arms.

Budgets differ per arm because they are budgeted in env-steps and JAX collects
far more per unit wall-clock; both are eval'd on the same wall-clock axis.

In [ ]:
# --- Editable config ---------------------------------------------------------
NUM_ENVS = 512            # JAX-only: vmapped envs per rollout (try 1024 / 2048)
ENV_STEPS_PER_UPDATE = 2  # matched update ratio applied to BOTH arms
JAX_BUDGET = 750_000      # env-steps for the Anakin (JAX) arm
SB3_BUDGET = 150_000      # env-steps for the SB3 (PyTorch/CPU) arm
SEEDS = 5                 # number of seeds per arm
LOG_DIR = "/content/results"

# Coupling (informational): updates_per_rollout = NUM_ENVS * ROLLOUT // ENV_STEPS_PER_UPDATE
# Scale NUM_ENVS up (1024 / 2048) to fill the TPU, raising ENV_STEPS_PER_UPDATE
# in step so updates/rollout stays sane.
print(f"NUM_ENVS={NUM_ENVS}  ENV_STEPS_PER_UPDATE={ENV_STEPS_PER_UPDATE}  "
      f"JAX_BUDGET={JAX_BUDGET}  SB3_BUDGET={SB3_BUDGET}  SEEDS={SEEDS}")

## 6. Run both arms across seeds

For each seed we run the Anakin (JAX) arm and the SB3 arm with the matched
`env_steps_per_update`, collecting each result dict. Each call is wrapped so a
single failing seed appends `None` and logs the error rather than aborting the
whole sweep. A per-seed progress line reports each arm's `time_to_solve`.

In [ ]:
import os
import traceback

from examples.rm.letterworld_anakin.train_anakin import run_anakin_dqn
from examples.rm.letterworld_anakin.train_sb3 import run_sb3_dqn

os.makedirs(LOG_DIR, exist_ok=True)

anakin_runs = []
sb3_runs = []

for seed in range(SEEDS):
    print(f"\n=== seed {seed} ===")

    try:
        a = run_anakin_dqn(
            JAX_BUDGET,
            seed,
            LOG_DIR,
            num_envs=NUM_ENVS,
            env_steps_per_update=ENV_STEPS_PER_UPDATE,
        )
        anakin_runs.append(a)
        a_tts = a["time_to_solve"]
    except Exception:  # noqa: BLE001
        print(f"[anakin] seed {seed} FAILED:")
        traceback.print_exc()
        anakin_runs.append(None)
        a_tts = None

    try:
        s = run_sb3_dqn(
            SB3_BUDGET,
            seed,
            LOG_DIR,
            env_steps_per_update=ENV_STEPS_PER_UPDATE,
        )
        sb3_runs.append(s)
        s_tts = s["time_to_solve"]
    except Exception:  # noqa: BLE001
        print(f"[sb3] seed {seed} FAILED:")
        traceback.print_exc()
        sb3_runs.append(None)
        s_tts = None

    print(f"seed {seed}: anakin time_to_solve={a_tts}  sb3 time_to_solve={s_tts}")

## 7. Plot: return vs wall-clock (the deliverable)

x = wall-clock seconds (`eval_wall`), y = mean eval return (`mean_return`), both
arms overlaid. For each arm we draw the per-seed curves as thin faint lines,
then build a **mean$\pm$std band** by interpolating every seed's
`(eval_wall, mean_return)` onto a **common wall-clock grid** with `np.interp`
before averaging. The x-axis is linear (a `plt.xscale("log")` line is provided,
commented out) — the JAX curve reaching high return far to the left is exactly
what we want to show.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ARMS = [
    ("Anakin (JAX/TPU)", anakin_runs, "#1f77b4"),
    ("SB3 (PyTorch/CPU)", sb3_runs, "#d62728"),
]
GRID_N = 200

fig, ax = plt.subplots(figsize=(9, 6))

for label, runs, color in ARMS:
    valid = [
        r for r in runs
        if r is not None and len(r["eval_wall"]) >= 2 and len(r["mean_return"]) >= 2
    ]
    if not valid:
        print(f"[plot] no valid seeds for {label}; skipping.")
        continue

    # Thin faint per-seed curves.
    for r in valid:
        ax.plot(
            r["eval_wall"], r["mean_return"],
            color=color, alpha=0.25, linewidth=1.0, zorder=1,
        )

    # Common wall-clock grid: up to the shortest run's final wall-clock so every
    # seed contributes across the whole grid (no extrapolation past its data).
    max_common_wall = min(r["eval_wall"][-1] for r in valid)
    if max_common_wall <= 0:
        print(f"[plot] degenerate wall-clock for {label}; skipping band.")
        continue
    grid = np.linspace(0.0, max_common_wall, GRID_N)

    interp = np.vstack([
        np.interp(grid, np.asarray(r["eval_wall"]), np.asarray(r["mean_return"]))
        for r in valid
    ])
    mean = interp.mean(axis=0)
    std = interp.std(axis=0)

    ax.plot(grid, mean, color=color, linewidth=2.5, zorder=3,
            label=f"{label} (mean of {len(valid)})")
    ax.fill_between(grid, mean - std, mean + std, color=color, alpha=0.18, zorder=2)

ax.set_xlabel("Wall-clock time (seconds)")
ax.set_ylabel("Mean eval return")
ax.set_title("LetterWorld DQN: return vs wall-clock (Anakin/JAX vs SB3), 5 seeds")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
# ax.set_xscale("log")  # optional: emphasize the low-wall-clock region

plt.savefig(
    "/content/letterworld_tpu_return_vs_time.png", dpi=120, bbox_inches="tight"
)
plt.show()

## 8. Summary

In [ ]:
import numpy as np


def _summarize(name, runs):
    valid = [r for r in runs if r is not None]
    solved = [r for r in valid if r.get("time_to_solve") is not None]
    if solved:
        mean_tts = float(np.mean([r["time_to_solve"] for r in solved]))
        mean_tts_str = f"{mean_tts:.1f}s"
    else:
        mean_tts_str = "n/a (none solved)"
    print(
        f"{name:20s} solved {len(solved)}/{SEEDS}  "
        f"mean time_to_solve (solved only): {mean_tts_str}"
    )


_summarize("Anakin (JAX/TPU)", anakin_runs)
_summarize("SB3 (PyTorch/CPU)", sb3_runs)

## Fairness

This comparison is deliberately **algorithmically matched** — the only thing
that differs is *execution*. The terms:

1. **`NUM_ENVS` scaling is JAX-only.** Running hundreds/thousands of vmapped
   environments per rollout is JAX's native execution model, not an advantage
   handed to it over SB3. SB3 steps a single (or vectorized-on-CPU) env in
   Python; it has no equivalent knob here.
2. **The update ratio is MATCHED on both arms.** `ENV_STEPS_PER_UPDATE` (and the
   resulting updates/rollout) is applied identically to Anakin and SB3. The same
   ratio is **cheap** for a jitted JAX kernel and **expensive** for SB3's
   PyTorch training loop — and that asymmetry, at equal work, *is the point*.
3. **Everything else is identical:** network architecture, discount $\gamma$,
   learning rate, batch size, replay-buffer size, target-update interval, the
   $\epsilon$-greedy schedule, and the eval protocol (every `K = 2000` env-steps
   over `N = 20` greedy episodes, "solved" at $\geq 0.95$ success).
4. **SB3 runs on the TPU VM's CPU.** Stable-Baselines3 uses PyTorch, which is
   **not** using the TPU here — it runs on the host CPU of the same Colab VM.
   This mirrors how each library is actually run in practice.